In [4]:
import json
from pathlib import Path
import sys

# Add project root to path to import forecaster
# Handle both cases: running from project root or from research directory
current_dir = Path().resolve()
if current_dir.name == 'research':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))

from forecaster import MatchData

# Load and serialize the JSON file
# Path relative to project root
json_file_path = project_root / 't20s_json' / '211028.json'

with open(json_file_path, 'r') as f:
    json_data = json.load(f)

# Convert to Python objects using Pydantic's model_validate
# Pydantic automatically handles nested structures!
match_data = MatchData.model_validate(json_data)

print(f"Match: {match_data.info.teams[0]} vs {match_data.info.teams[1]}")
print(f"Date: {match_data.info.dates[0]}")
print(f"Venue: {match_data.info.venue}")
print(f"Winner: {match_data.info.outcome.winner}")
print(f"Number of innings: {len(match_data.innings)}")
print(f"Total overs in first innings: {len(match_data.innings[0].overs)}")

# Pydantic benefits: automatic validation and easy serialization back to JSON
print(f"\n--- Pydantic Benefits ---")
print(f"Model validation: ✓")
print(f"Can serialize back: {match_data.model_dump_json()[:100]}...")


Match: England vs Australia
Date: 2005-06-13
Venue: The Rose Bowl
Winner: England
Number of innings: 2
Total overs in first innings: 20

--- Pydantic Benefits ---
Model validation: ✓
Can serialize back: {"meta":{"data_version":"1.0.0","created":"2013-02-22","revision":1},"info":{"balls_per_over":6,"cit...


In [5]:
# Display JSON Schema Structure
import json
from pathlib import Path

# Load the schema template
schema_path = project_root / 'forecaster' / 'json_schema_template.json'

with open(schema_path, 'r') as f:
    schema = json.load(f)

# Pretty print the schema
print("=" * 80)
print("JSON STRUCTURE SCHEMA (without actual data)")
print("=" * 80)
print(json.dumps(schema, indent=2))


JSON STRUCTURE SCHEMA (without actual data)
{
  "meta": {
    "data_version": "string",
    "created": "string (YYYY-MM-DD)",
    "revision": "integer"
  },
  "info": {
    "balls_per_over": "integer (typically 6)",
    "city": "string",
    "dates": [
      "string (YYYY-MM-DD)"
    ],
    "gender": "string (e.g., 'male', 'female')",
    "match_type": "string (e.g., 'T20')",
    "match_type_number": "integer",
    "officials": {
      "match_referees": [
        "string"
      ],
      "tv_umpires": [
        "string"
      ],
      "umpires": [
        "string"
      ],
      "reserve_umpires": [
        "string (optional)"
      ]
    },
    "outcome": {
      "by": {
        "runs": "integer (optional)",
        "wickets": "integer (optional)"
      },
      "winner": "string (team name)"
    },
    "overs": "integer (typically 20 for T20)",
    "player_of_match": [
      "string (optional)"
    ],
    "players": {
      "TeamName1": [
        "string (player names)"
      ],
     

In [6]:
# Break match_data into two innings and calculate scores

# Extract the two innings
first_innings = match_data.innings[0]
second_innings = match_data.innings[1]

# Function to calculate innings score
def calculate_innings_score(innings):
    """Calculate total runs and wickets for an innings"""
    total_runs = 0
    total_wickets = 0
    
    for over in innings.overs:
        for delivery in over.deliveries:
            total_runs += delivery.runs.total
            if delivery.wickets:
                total_wickets += len(delivery.wickets)
    
    return total_runs, total_wickets

# Calculate scores for both innings
first_innings_runs, first_innings_wickets = calculate_innings_score(first_innings)
second_innings_runs, second_innings_wickets = calculate_innings_score(second_innings)

# Display the scores
print("=" * 80)
print("MATCH SCORECARD")
print("=" * 80)
print(f"\n{first_innings.team}: {first_innings_runs}/{first_innings_wickets} ({len(first_innings.overs)} overs)")
print(f"{second_innings.team}: {second_innings_runs}/{second_innings_wickets} ({len(second_innings.overs)} overs)")

# Show margin of victory
if match_data.info.outcome.by.get('runs'):
    margin = match_data.info.outcome.by['runs']
    print(f"\n{match_data.info.outcome.winner} won by {margin} runs")
elif match_data.info.outcome.by.get('wickets'):
    margin = match_data.info.outcome.by['wickets']
    print(f"\n{match_data.info.outcome.winner} won by {margin} wickets")

print("\n" + "=" * 80)


MATCH SCORECARD

England: 179/8 (20 overs)
Australia: 79/10 (15 overs)

England won by 100 runs



In [7]:
# Additional statistics for each innings

def get_innings_statistics(innings):
    """Get detailed statistics for an innings"""
    stats = {
        'team': innings.team,
        'total_runs': 0,
        'total_wickets': 0,
        'total_balls': 0,
        'fours': 0,
        'sixes': 0,
        'extras': 0,
        'byes': 0,
        'legbyes': 0,
        'wides': 0,
        'noballs': 0
    }
    
    for over in innings.overs:
        for delivery in over.deliveries:
            stats['total_runs'] += delivery.runs.total
            stats['total_balls'] += 1
            
            # Count boundaries
            if delivery.runs.batter == 4:
                stats['fours'] += 1
            elif delivery.runs.batter == 6:
                stats['sixes'] += 1
            
            # Count wickets
            if delivery.wickets:
                stats['total_wickets'] += len(delivery.wickets)
            
            # Count extras
            if delivery.extras:
                stats['extras'] += delivery.runs.extras
                if 'byes' in delivery.extras:
                    stats['byes'] += delivery.extras['byes']
                if 'legbyes' in delivery.extras:
                    stats['legbyes'] += delivery.extras['legbyes']
                if 'wides' in delivery.extras:
                    stats['wides'] += delivery.extras['wides']
                if 'noballs' in delivery.extras:
                    stats['noballs'] += delivery.extras['noballs']
    
    # Calculate run rate
    stats['run_rate'] = (stats['total_runs'] / stats['total_balls'] * 6) if stats['total_balls'] > 0 else 0
    
    return stats

# Get statistics for both innings
first_innings_stats = get_innings_statistics(first_innings)
second_innings_stats = get_innings_statistics(second_innings)

# Display detailed statistics
print("\n" + "=" * 80)
print("DETAILED INNINGS STATISTICS")
print("=" * 80)

for stats in [first_innings_stats, second_innings_stats]:
    print(f"\n{stats['team']}:")
    print(f"  Total: {stats['total_runs']}/{stats['total_wickets']} in {stats['total_balls']} balls")
    print(f"  Run Rate: {stats['run_rate']:.2f}")
    print(f"  Boundaries: {stats['fours']} fours, {stats['sixes']} sixes")
    print(f"  Extras: {stats['extras']} ({stats['wides']} wides, {stats['noballs']} no-balls, "
          f"{stats['byes']} byes, {stats['legbyes']} leg-byes)")

print("\n" + "=" * 80)



DETAILED INNINGS STATISTICS

England:
  Total: 179/8 in 125 balls
  Run Rate: 8.59
  Boundaries: 19 fours, 3 sixes
  Extras: 6 (3 wides, 2 no-balls, 0 byes, 1 leg-byes)

Australia:
  Total: 79/10 in 90 balls
  Run Rate: 5.27
  Boundaries: 10 fours, 0 sixes
  Extras: 6 (1 wides, 2 no-balls, 1 byes, 2 leg-byes)

